## Import the Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import matplotlib.pyplot as plt

## Load The Dataset

In [2]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(df.head())

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  


# Feature engineering

In [3]:
print(df.shape)

print(df.info())

print(df.describe())

(20640, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900      1.000000      0.846154      0.333

# Test/Train Split

In [4]:
X = df.drop("MedHouseVal", axis=1)

y = df["MedHouseVal"]

In [5]:
corr = df.corr()

print(corr["MedHouseVal"].sort_values(
    ascending=False
))

MedHouseVal    1.000000
MedInc         0.688075
AveRooms       0.151948
HouseAge       0.105623
AveOccup      -0.023737
Population    -0.024650
Longitude     -0.045967
AveBedrms     -0.046701
Latitude      -0.144160
Name: MedHouseVal, dtype: float64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Base Line Model

In [7]:
tree = DecisionTreeRegressor(
    random_state=42
)

tree.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

In [8]:
y_pred = tree.predict(X_test)

# Test Scores

In [15]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=42)

tree.fit(X_train, y_train)

print("Train R²:", tree.score(X_train, y_train))
print("Test R² :", tree.score(X_test, y_test))

Train R²: 1.0
Test R² : 0.622075845135081


Here you can see overfitting so we need to do the pruning

In [16]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

      Feature  Importance
0      MedInc    0.528509
5    AveOccup    0.130838
6    Latitude    0.093717
7   Longitude    0.082902
2    AveRooms    0.052975
1    HouseAge    0.051884
4  Population    0.030516
3   AveBedrms    0.028660


# Pruning

In [17]:
print("Depth:", tree.get_depth())
print("Leaves:", tree.get_n_leaves())

Depth: 34
Leaves: 15854


In [32]:
tree = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=15,
    min_samples_leaf=8,
    random_state=42
)

tree.fit(X_train, y_train)

print("Train:", tree.score(X_train, y_train))
print("Test :", tree.score(X_test, y_test))

Train: 0.8102485549622706
Test : 0.7105316793662346


# Let's find the Best One

In [33]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

def find_best_tree(
    X_train,
    X_test,
    y_train,
    y_test,
    max_depths,
    min_samples_splits,
    min_samples_leafs,
    ccp_alphas
):

    best_score = float("-inf")
    best_model = None
    best_params = None

    results = []

    for depth in max_depths:

        for split in min_samples_splits:

            for leaf in min_samples_leafs:

                for alpha in ccp_alphas:

                    model = DecisionTreeRegressor(
                        max_depth=depth,
                        min_samples_split=split,
                        min_samples_leaf=leaf,
                        ccp_alpha=alpha,
                        random_state=42
                    )

                    model.fit(X_train, y_train)

                    train_r2 = model.score(X_train, y_train)
                    test_r2 = model.score(X_test, y_test)

                    results.append({
                        "max_depth": depth,
                        "min_samples_split": split,
                        "min_samples_leaf": leaf,
                        "ccp_alpha": alpha,
                        "train_r2": train_r2,
                        "test_r2": test_r2
                    })

                    if test_r2 > best_score:
                        best_score = test_r2
                        best_model = model
                        best_params = {
                            "max_depth": depth,
                            "min_samples_split": split,
                            "min_samples_leaf": leaf,
                            "ccp_alpha": alpha
                        }

    return best_model, best_params, best_score, results

In [34]:
best_model, best_params, best_score, results = find_best_tree(
    X_train,
    X_test,
    y_train,
    y_test,

    max_depths=[3, 5, 8, 10, 15],

    min_samples_splits=[2, 5, 10, 20],

    min_samples_leafs=[1, 2, 5, 10],

    ccp_alphas=[0.0, 0.0001, 0.001, 0.01]
)

In [35]:
print("Best Parameters:")
print(best_params)

print("\nBest Test R²:")
print(best_score)

Best Parameters:
{'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 10, 'ccp_alpha': 0.0001}

Best Test R²:
0.7233186277658732


In [36]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="test_r2",
    ascending=False
)

print(results_df.head(10))

     max_depth  min_samples_split  min_samples_leaf  ccp_alpha  train_r2  \
301         15                 10                10     0.0001  0.835747   
285         15                  5                10     0.0001  0.835747   
269         15                  2                10     0.0001  0.835747   
317         15                 20                10     0.0001  0.835747   
284         15                  5                10     0.0000  0.854125   
268         15                  2                10     0.0000  0.854125   
300         15                 10                10     0.0000  0.854125   
316         15                 20                10     0.0000  0.854125   
313         15                 20                 5     0.0001  0.854031   
312         15                 20                 5     0.0000  0.873061   

      test_r2  
301  0.723319  
285  0.723319  
269  0.723319  
317  0.723319  
284  0.722732  
268  0.722732  
300  0.722732  
316  0.722732  
313  0.721768  
312